In [1]:
import pandas as pd
import numpy as np

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.preprocess_tools import decouple_streams

# Binary classes rolling window features

Window size is 2 seconds.
Overlap is 50%.
Feature label is decided by majority vote.

In [2]:
labels_df = pd.read_csv("../data/labels/sprint_start_end.csv")

In [4]:
window_size = 2000
window_dist = 1000

processed_list = []

for trial_num, label in labels_df.iterrows():
    raw_accel, clean_accel, gyro = decouple_streams(pd.read_csv(label["path"]))

    last_viable_time = min(
        raw_accel["timestamp"].iloc[-1],
        clean_accel["timestamp"].iloc[-1],
        gyro["timestamp"].iloc[-1],
    )

    for window_start in range(0, last_viable_time - window_size + 1, window_dist):
        window_end = window_start + window_size
        window_mid = window_start + window_size // 2

        window_label = (
            1 if ((window_mid > label["start"]) & (window_mid < label["end"])) else 0
        )

        window_feature = {
            "trial_num": trial_num,
            "start": window_start,
            "end": window_end,
            "sprint": window_label,
        }

        raw_accel_window = raw_accel[
            raw_accel["timestamp"].between(window_start, window_end, inclusive="left")
        ]
        clean_accel_window = clean_accel[
            clean_accel["timestamp"].between(window_start, window_end, inclusive="left")
        ]
        gyro_window = gyro[
            gyro["timestamp"].between(window_start, window_end, inclusive="left")
        ]

        prefixes = ["raw_accel_", "clean_accel_", "gyro_"]
        windows = (raw_accel_window, clean_accel_window, gyro_window)

        for prefix, window in zip(prefixes, windows):
            magnitude = np.sqrt(window["x"] ** 2 + window["y"] ** 2 + window["z"] ** 2)

            window_feature[f"{prefix}mean"] = magnitude.mean()
            window_feature[f"{prefix}max"] = magnitude.max()
            window_feature[f"{prefix}std"] = magnitude.std()
            window_feature[f"{prefix}rms"] = np.sqrt(np.mean(magnitude**2))

            if prefix in ["clean_accel_", "gyro_"]:
                for axis in ["x", "y", "z"]:
                    signal = window[axis] - window[axis].mean()

                    crossings = ((signal.shift(1) < 0) & (signal >= 0)) | (
                        (signal.shift(1) >= 0) & (signal < 0)
                    )

                    window_feature[f"{prefix}zcr_{axis}"] = crossings.dropna().sum() / (
                        len(signal) - 1
                    )

        processed_list.append(window_feature)

processed_df = pd.DataFrame(processed_list)
processed_df.to_csv("../data/features/engineered_features_v1.csv")